# Student Performance Prediction & Segmentation

Analysing 1,044 student records to predict academic outcomes and identify actionable learner segments using supervised and unsupervised machine learning.

**Approach:**
- Random Forest Regression → predict final grades and identify key performance drivers
- K-Means Clustering → segment students into intervention-ready profiles
- Categorical Analysis → surface contextual patterns from qualitative variables

**Dataset:** [UCI Student Performance](https://archive.ics.uci.edu/ml/datasets/student+performance) (Cortez & Silva, 2008) — 1,044 records, 34 features

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 150

In [ ]:
df = pd.read_csv('../data/student_performance.csv')

print(f"Shape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")
df.head()

## 2. Exploratory Data Analysis

Before modelling, we examine the target variable distribution and feature correlations to understand the data landscape.

In [ ]:
print("G3 (Final Grade) Statistics:")
print(df['G3'].describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# G3 distribution
axes[0].hist(df['G3'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Final Grades (G3)')
axes[0].set_xlabel('Final Grade (G3)')
axes[0].set_ylabel('Number of Students')
axes[0].axvline(df['G3'].mean(), color='red', linestyle='--',
                label=f"Mean = {df['G3'].mean():.2f}")
axes[0].legend()

# Correlation heatmap
numeric_cols = ['age', 'Medu', 'Fedu', 'studytime', 'failures',
                'absences', 'G1', 'G2', 'G3']
correlation = df[numeric_cols].corr()
sns.heatmap(correlation, ax=axes[1], cmap='RdYlGn',
            annot=True, fmt='.2f', linewidths=0.3)
axes[1].set_title('Correlation Between Key Features')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../outputs/fig1_G3_Distribution.png', dpi=150)
plt.show()

**Key observations:**
- G3 is bimodal — the spike at 0 represents student withdrawal, not academic failure
- G1→G2→G3 are highly correlated (r=0.81–0.91), confirming grade continuity
- Parental education shows moderate positive correlation with G3 (r≈0.20)
- Prior failures inversely correlate with final grades (r=-0.38)

## 3. Preprocessing

In [ ]:
df_enc = df.copy()

# Binary encoding
binary_map = {
    'sex': {'F': 0, 'M': 1}, 'address': {'U': 1, 'R': 0},
    'famsize': {'LE3': 0, 'GT3': 1}, 'Pstatus': {'T': 1, 'A': 0},
    'schoolsup': {'yes': 1, 'no': 0}, 'famsup': {'yes': 1, 'no': 0},
    'paid': {'yes': 1, 'no': 0}, 'activities': {'yes': 1, 'no': 0},
    'nursery': {'yes': 1, 'no': 0}, 'higher': {'yes': 1, 'no': 0},
    'internet': {'yes': 1, 'no': 0}, 'romantic': {'yes': 1, 'no': 0},
}

for col, mapping in binary_map.items():
    df_enc[col] = df_enc[col].map(mapping)

# Label encoding for nominal variables
# Acceptable for tree-based models which split on thresholds, not magnitude
le = LabelEncoder()
for col in ['school', 'Mjob', 'Fjob', 'reason', 'guardian', 'course']:
    df_enc[col] = le.fit_transform(df_enc[col])

print("Encoding complete.")
df_enc[['sex', 'Mjob', 'reason', 'higher']].head()

## 4. Supervised Learning — Random Forest Regression

Predicting final grade (G3) using all available features. Random Forest was chosen for its ability to handle mixed feature types, resist overfitting through bagging, and provide native feature importance rankings.

In [ ]:
# Train-test split
X = df_enc.drop(columns=['G3'])
y = df_enc['G3']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Random Forest
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

# Evaluate
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Random Forest Results:")
print(f"  R²   = {r2:.4f}")
print(f"  RMSE = {rmse:.4f}")
print(f"  MAE  = {mae:.4f}")

In [ ]:
# Baseline comparison — Linear Regression
baseline = LinearRegression()
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

baseline_r2 = r2_score(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))

print(f"Linear Regression Baseline:  R² = {baseline_r2:.4f}  |  RMSE = {baseline_rmse:.4f}")
print(f"Random Forest:               R² = {r2:.4f}  |  RMSE = {rmse:.4f}")
print(f"\nRF improvement:  R² +{r2 - baseline_r2:.4f}  |  RMSE -{baseline_rmse - rmse:.4f}")

In [ ]:
# Cross-validation
cv_scores = cross_val_score(rf, X, y, cv=5, scoring='r2')
print(f"5-Fold CV R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

In [ ]:
# Feature importance
importances = pd.Series(rf.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=True).tail(15)

plt.figure(figsize=(10, 6))
importances.plot(kind='barh', color='steelblue')
plt.title('Top 15 Predictive Features — Random Forest')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('../outputs/fig2_Relevant_Categories.png', dpi=150)
plt.show()

**Takeaway:** G2 dominates the feature importance rankings (~82% of predictive weight). While statistically valid, this limits practical utility — for intervention design, modifiable factors like `absences` and `studytime` are more actionable.

## 5. Unsupervised Learning — K-Means Clustering

Segmenting students based on 14 modifiable behavioural features. G3 is deliberately excluded from clustering to avoid target leakage, then used post-hoc as external validation.

In [ ]:
# Select modifiable features only
modifiable_features = [
    'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities',
    'internet', 'absences', 'goout', 'Dalc', 'Walc', 'famrel', 'freetime', 'higher'
]

X_cluster = df_enc[modifiable_features]

# Standardise — essential for distance-based algorithms
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

In [ ]:
# Find optimal k using Elbow Method + Silhouette Score
inertias = []
silhouettes = []
k_range = range(2, 9)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(list(k_range), inertias, 'bo-', linewidth=2)
axes[0].axvline(4, color='red', linestyle='--', label='Selected k=4')
axes[0].set_title('Elbow Method')
axes[0].set_xlabel('Number of Clusters')
axes[0].set_ylabel('Inertia (SSE)')
axes[0].legend()

axes[1].plot(list(k_range), silhouettes, 'gs-', linewidth=2)
axes[1].axvline(4, color='red', linestyle='--', label='Selected k=4')
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('Number of Clusters')
axes[1].set_ylabel('Silhouette Score')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/fig3_Optimal_Groups.png', dpi=150)
plt.show()

In [ ]:
# Fit final model with k=4
final_km = KMeans(n_clusters=4, random_state=42, n_init=10)
df_enc['cluster'] = final_km.fit_predict(X_scaled)

# Label clusters by ascending mean G3
g3_ranking = df_enc.groupby('cluster')['G3'].mean().sort_values()
label_map = {
    g3_ranking.index[0]: 'Low Achievers',
    g3_ranking.index[1]: 'Struggling Students',
    g3_ranking.index[2]: 'Average Students',
    g3_ranking.index[3]: 'High Achievers'
}
df_enc['student_group'] = df_enc['cluster'].map(label_map)

print("Cluster sizes:")
print(df_enc['student_group'].value_counts())
print(f"\nSilhouette Score: {silhouette_score(X_scaled, df_enc['cluster']):.3f}")
print("\nMean G3 per cluster:")
print(df_enc.groupby('student_group')['G3'].mean().round(2))

In [ ]:
# Cluster profile visualisation
group_order = ['Low Achievers', 'Struggling Students', 'Average Students', 'High Achievers']
group_colours = {
    'Low Achievers': '#d73027', 'Struggling Students': '#fc8d59',
    'Average Students': '#4575b4', 'High Achievers': '#1a9641'
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Student Group Profiles', fontweight='bold')

# G3 boxplot per group
box_data = [df_enc[df_enc['student_group'] == g]['G3'].values for g in group_order]
bp = axes[0].boxplot(box_data, labels=group_order, patch_artist=True)
for patch, group in zip(bp['boxes'], group_order):
    patch.set_facecolor(group_colours[group])
axes[0].set_title('Final Grade by Student Group')
axes[0].set_ylabel('Final Grade (G3)')
axes[0].tick_params(axis='x', rotation=15)

# Key feature comparison
key_features = ['studytime', 'failures', 'absences', 'goout', 'Dalc']
x = np.arange(len(key_features))
width = 0.2

for i, group in enumerate(group_order):
    means = df_enc[df_enc['student_group'] == group][key_features].mean()
    axes[1].bar(x + i * width, means, width=width,
                label=group, color=group_colours[group], alpha=0.85)

axes[1].set_xticks(x + width * 1.5)
axes[1].set_xticklabels(key_features, rotation=20)
axes[1].set_title('Key Feature Comparison by Group')
axes[1].set_ylabel('Mean Value')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('../outputs/fig4_Student_Group_Profiles.png', dpi=150)
plt.show()

## 6. Categorical Analysis

Frequency and mean-grade cross-tabulation of qualitative variables to identify contextual patterns that numeric modelling alone would miss.

In [ ]:
for col in ['reason', 'guardian', 'Mjob']:
    summary = pd.DataFrame({
        'Count': df[col].value_counts(),
        'Mean G3': df.groupby(col)['G3'].mean().round(2)
    }).sort_values('Mean G3', ascending=False)
    print(f"\n{'='*40}")
    print(f"  {col.upper()}")
    print(f"{'='*40}")
    print(summary)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Qualitative Features vs Final Grade (G3)', fontweight='bold')

qual_features = [
    ('reason', 'School Choice Reason'),
    ('guardian', 'Guardian Type'),
    ('Mjob', 'Mother Occupation'),
    ('higher', 'Higher Education Aspiration')
]

for ax, (col, title) in zip(axes.flat, qual_features):
    order = df.groupby(col)['G3'].mean().sort_values(ascending=False).index.tolist()
    sns.boxplot(data=df, x=col, y='G3', order=order, ax=ax, palette='Set2', width=0.5)
    ax.set_title(title)
    ax.set_xlabel(col)
    ax.set_ylabel('Final Grade (G3)')
    ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('../outputs/fig5_Qualitative_Analysis.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Category Frequency and Mean Grade Analysis', fontweight='bold')

for ax, col in zip(axes, ['reason', 'guardian', 'Mjob']):
    counts = df[col].value_counts()
    means = df.groupby(col)['G3'].mean().reindex(counts.index)

    ax2 = ax.twinx()
    ax.bar(counts.index, counts.values, color='steelblue', alpha=0.65, label='Frequency')
    ax2.plot(counts.index, means.values, 'ro-', linewidth=2, markersize=7, label='Mean G3')

    ax.set_title(f'{col}')
    ax.set_ylabel('Count', color='steelblue')
    ax2.set_ylabel('Mean G3', color='red')
    ax2.tick_params(axis='y', labelcolor='red')
    ax.tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.savefig('../outputs/fig6_Frequency_Analysis.png', dpi=150)
plt.show()

**Key findings:**
- Students aspiring to higher education scored **3+ grade points higher** on average (11.62 vs 8.35)
- Maternal occupation in health/teaching correlates with higher grades (12.68 and 12.21 respectively)
- Reputation-motivated students outperform course-motivated ones (12.18 vs 10.97)

## 7. Summary

| Component | Method | Key Result |
|-----------|--------|------------|
| Performance prediction | Random Forest Regression | R²=0.82, RMSE=1.65 |
| Baseline comparison | Linear Regression | R²=0.80 (RF outperforms) |
| Student segmentation | K-Means (k=4) | 4 distinct learner profiles with clear G3 separation |
| Categorical analysis | Frequency + mean-grade cross-tabulation | Higher education aspiration = strongest contextual differentiator |

**Limitations:**
- G2 dominates feature importance, reducing actionable insight from the supervised model
- Silhouette score of 0.125 indicates soft cluster boundaries
- Dataset is from Portuguese secondary schools (2005–2006); context-specific transferability is limited

**Potential extensions:**
- Gaussian Mixture Models for probabilistic cluster assignment
- LDA topic modelling on genuine free-text student data
- Deploying the model as an API or interactive dashboard